In [ ]:
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
import re
import gc
import ast
import sys
import io
import json
import contextlib
import traceback
import resource
import threading
import ctypes
import concurrent.futures
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
class TimeoutException(Exception):
    pass


def _async_raise_in_thread(tid, exc_type):
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(
        ctypes.c_ulong(tid), ctypes.py_object(exc_type)
    )
    if res == 0:
        return False
    if res > 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(ctypes.c_ulong(tid), None)
        return False
    return True


def analyze_assertion_failure(line_code, context):
    try:
        tree = ast.parse(line_code.strip())
        if not isinstance(tree.body[0], ast.Assert):
            return None
        assert_node = tree.body[0]
        node = assert_node.test
        if isinstance(node, ast.Compare):
            left_node = node.left
            right_node = node.comparators[0]
            left_val = eval(compile(ast.Expression(body=left_node), filename="<string>", mode="eval"), context)
            right_val = eval(compile(ast.Expression(body=right_node), filename="<string>", mode="eval"), context)
            return {
                "actual": left_val,
                "expected": right_val,
                "actual_repr": ast.unparse(left_node),
                "expected_repr": ast.unparse(right_node),
            }
    except Exception:
        return None
    return None


def execute_code_with_feedback(code_str, cases=None, timeout_seconds=5, memory_limit_mb=4096):
    if cases is None:
        cases = []
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()
    full_execution_script = f"{code_str}\n\n" + "\n".join(cases)

    target_tid = threading.get_ident()
    timer = threading.Timer(
        timeout_seconds,
        lambda: _async_raise_in_thread(target_tid, TimeoutException),
    )
    timer.daemon = True
    timer.start()

    apply_mem_limit = threading.current_thread() is threading.main_thread()
    soft = hard = None
    if apply_mem_limit:
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        try:
            resource.setrlimit(resource.RLIMIT_AS, (memory_limit_mb * 1024 * 1024, hard))
        except ValueError:
            apply_mem_limit = False

    local_scope = {}
    try:
        with contextlib.redirect_stdout(stdout_capture), contextlib.redirect_stderr(stderr_capture):
            exec(full_execution_script, local_scope)
        timer.cancel()
        return True, f"Execution Successful. Stdout:\n{stdout_capture.getvalue()}", "None"
    except TimeoutException:
        timer.cancel()
        return False, "Execution Failed: Time Limit Exceeded (Possible Infinite Loop)", "Runtime"
    except MemoryError:
        timer.cancel()
        return False, "Execution Failed: Memory Limit Exceeded", "Runtime"
    except Exception:
        timer.cancel()
        exc_type, exc_value, exc_traceback = sys.exc_info()
        exc_name = exc_type.__name__
        tb_frames = traceback.extract_tb(exc_traceback)
        error_line_num = "Unknown"
        offending_line_code = "Could not extract line."
        if hasattr(exc_value, "lineno"):
            error_line_num = exc_value.lineno
        elif tb_frames:
            for frame in reversed(tb_frames):
                if frame.filename == "<string>":
                    error_line_num = frame.lineno
                    break
        if isinstance(error_line_num, int):
            script_lines = full_execution_script.splitlines()
            if 0 <= error_line_num - 1 < len(script_lines):
                offending_line_code = script_lines[error_line_num - 1].strip()
        details = str(exc_value)
        rich_feedback = ""
        if exc_name == "AssertionError":
            analysis = analyze_assertion_failure(offending_line_code, local_scope)
            if analysis:
                act_str = str(analysis["actual"])
                exp_str = str(analysis["expected"])
                if len(act_str) > 500: act_str = act_str[:500] + "... (truncated)"
                if len(exp_str) > 500: exp_str = exp_str[:500] + "... (truncated)"
                rich_feedback = (
                    f"\nAnalysis:\n   Input/Call: {analysis['actual_repr']}\n"
                    f"   Expected: {exp_str}\n   Actual:    {act_str}"
                )
        error_msg = f"Error Type: {exc_name}\nLine {error_line_num}: {offending_line_code}\nDetails: {details}{rich_feedback}"
        return False, error_msg, "Logical" if exc_name == "AssertionError" else "Runtime"
    finally:
        timer.cancel()
        if apply_mem_limit and soft is not None:
            try:
                resource.setrlimit(resource.RLIMIT_AS, (soft, hard))
            except ValueError:
                pass

In [ ]:
import huggingface_hub

huggingface_hub.login()

In [ ]:
MODEL_NAME  = "moazeldegwy/Qwen3-0.6B-LABD-2.1-SFT"
MAX_SEQ_LEN = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = False,
)

FastLanguageModel.for_inference(model)

In [ ]:
TEMPERATURE         = 0.6
TOP_P               = 0.95
TOP_K               = 20
MIN_P               = 0.0
REPETITION_PENALTY  = 1.05

SYSTEM_PROMPT = """You are a Self-Correcting Python Agent. Solve the problem; the sandbox tool will verify your code.
### OUTPUT FORMAT
After your reasoning, output ONE <execute> block containing the function plus the user's assert statements appended verbatim. Nothing else after it.
Example:
<execute>
def add(a, b):
    return a + b
assert add(2, 3) == 5
</execute>
### RULES
- Exactly one <execute> block per turn. Function and asserts together, never split into separate blocks.
- On a debugging turn: quote Expected vs Actual from the sandbox tool feedback in your reasoning and identify the root cause before writing the corrected <execute> block.
"""


class BatchedSelfCorrectionAgent:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

        self.tokenizer.padding_side = "left"
        self.tokenizer.truncation_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.sys_instruction = SYSTEM_PROMPT

    def extract_code(self, response_text):
        code_match = re.search(r"<execute>(.*?)</execute>", response_text, re.DOTALL)
        final_markdown = re.search(r"```python(.*?)```", response_text, re.DOTALL)

        code_to_run = None
        if code_match:
            code_to_run = code_match.group(1).strip()
        elif final_markdown:
            code_to_run = final_markdown.group(1).strip()

        if code_to_run:
            if code_to_run.startswith("```python"): code_to_run = code_to_run.replace("```python", "", 1)
            if code_to_run.startswith("```"):       code_to_run = code_to_run.replace("```", "", 1)
            if code_to_run.endswith("```"):         code_to_run = code_to_run[:-3]
            return code_to_run.strip()
        return None

    def run_batched(self, dataset, batch_size=8, max_retries=3, output_file="batched_results.jsonl"):
        processed_ids = set()
        if os.path.exists(output_file):
            with open(output_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        record = json.loads(line)
                        if "task_id" in record:
                            processed_ids.add(record["task_id"])
                    except json.JSONDecodeError:
                        pass
            print(f"Resuming... {len(processed_ids)} tasks already processed.")

        active_tasks = [item for item in dataset if item["task_id"] not in processed_ids]
        total_tasks = len(active_tasks)
        print(f"Starting Batched Processing: {total_tasks} tasks to do (Batch Size: {batch_size})")

        num_batches = (total_tasks + batch_size - 1) // batch_size
        for i in range(0, total_tasks, batch_size):
            batch = active_tasks[i:i + batch_size]
            print(f"\nProcessing Batch {i // batch_size + 1} / {num_batches}...")
            self._process_single_batch(batch, max_retries, output_file)

    def _process_single_batch(self, batch, max_retries, output_file):
        MAX_USER_TOKENS = 1800
        states = []
        for task in batch:
            raw_user_msg = f"Problem: {task['prompt']}\n\nTest Cases:\n" + "\n".join(task["tests"])
            user_msg_tokens = self.tokenizer.encode(raw_user_msg, add_special_tokens=False)
            if len(user_msg_tokens) > MAX_USER_TOKENS:
                user_msg_tokens = user_msg_tokens[:MAX_USER_TOKENS]
                safe_user_msg = self.tokenizer.decode(user_msg_tokens) + "\n# [WARNING: Test cases truncated due to length]"
            else:
                safe_user_msg = raw_user_msg

            states.append({
                "task_id": task["task_id"],
                "tests": task["tests"],
                "messages": [
                    {"role": "system", "content": self.sys_instruction},
                    {"role": "user",   "content": safe_user_msg},
                ],
                "attempts": 0,
                "solved": False,
                "done": False,
            })

        while True:
            active_states = [s for s in states if not s["done"]]
            if not active_states:
                break

            texts = [
                self.tokenizer.apply_chat_template(
                    s["messages"],
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=True,
                )
                for s in active_states
            ]

            inputs = self.tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=6144,
            ).to("cuda")

            with torch.inference_mode():
                generated_ids = self.model.generate(
                    **inputs,
                    max_new_tokens     = 6144,
                    do_sample          = True,
                    temperature        = TEMPERATURE,
                    top_p              = TOP_P,
                    top_k              = TOP_K,
                    min_p              = MIN_P,
                    repetition_penalty = REPETITION_PENALTY,
                    pad_token_id       = self.tokenizer.pad_token_id,
                )

            new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
            responses = self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            executor_results = []
            with concurrent.futures.ThreadPoolExecutor(max_workers=max(1, len(active_states))) as executor:
                futures = []
                for idx, response in enumerate(responses):
                    code = self.extract_code(response)
                    if code:
                        futures.append(executor.submit(execute_code_with_feedback, code, active_states[idx]["tests"]))
                    else:
                        futures.append(None)

                for f in futures:
                    if f:
                        executor_results.append(f.result())
                    else:
                        executor_results.append((False, "Error: No valid code block found inside <execute> tags.", "Parsing"))

            for idx, (success, output_msg, error_type) in enumerate(executor_results):
                state = active_states[idx]
                state["attempts"] += 1
                state["messages"].append({"role": "assistant", "content": responses[idx]})

                if success:
                    print(f"  > Task {state['task_id']} Attempt {state['attempts']}: SUCCESS")
                    state["solved"] = True
                    state["done"]   = True
                    state["messages"].append({"role": "user", "content": f"<tool_response>\nExecution Success\n{output_msg}\n</tool_response>"})
                else:
                    print(f"  > Task {state['task_id']} Attempt {state['attempts']}: FAILED ({error_type})")
                    state["messages"].append({"role": "user", "content": f"<tool_response>\nExecution Failed\n{output_msg}\n</tool_response>"})
                    if state["attempts"] >= max_retries:
                        state["done"] = True

                if state["done"]:
                    result_entry = {
                        "task_id": state["task_id"],
                        "solved": state["solved"],
                        "attempts_used": state["attempts"],
                        "conversation": state["messages"],
                    }
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(json.dumps(result_entry) + "\n")

            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset = load_dataset("google-research-datasets/mbpp")
test_data = dataset["test"].to_list()
test_data = test_data[:165]

prepared_data = []
for row in test_data:
    prepared_data.append({
        "task_id": row["task_id"],
        "prompt":  row["text"],
        "tests":   row["test_list"],
    })

print(f"Loaded {len(prepared_data)} tasks from MBPP test split.")

drive_output_path = "/content/drive/MyDrive/LABD2/MBPP_SFT_qwen3_0.6B_labd_2.1.jsonl"

agent = BatchedSelfCorrectionAgent(model, tokenizer)
agent.run_batched(
    prepared_data,
    batch_size=10,
    max_retries=3,
    output_file=drive_output_path,
)

In [ ]:
records = []
with open(drive_output_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        records.append(json.loads(line))

total   = len(records)
solved  = sum(1 for r in records if r["solved"])
by_attempts = {1: 0, 2: 0, 3: 0}
for r in records:
    if r["solved"] and r["attempts_used"] in by_attempts:
        by_attempts[r["attempts_used"]] += 1

print(f"Total tasks: {total}")
print(f"Solved:      {solved} ({solved / max(total, 1):.1%})")
for k, v in by_attempts.items():
    print(f"  solved on attempt {k}: {v}")